# Football3D — HMR2 for the SNGS-043 goal sequence

Pose only three stable players during 00:00:15–00:00:25, ending at the goal: frames 375–625, tracks 284, 17, and 171. They remain near the goal at frame 625 and cover 250, 248, and 221 of the 251 frames. Attach the SoccerNet frames + raw track JSON and private SMPL input. Enable Internet and a T4 GPU.

**HMR2 cache (2.7 GB):** first run downloads and extracts it to `/kaggle/working/hmr2_cache`. Save the completed version, make a **private** Kaggle Dataset from that output folder, and attach it to later runs; then cell 3 finds it and skips the download. The cache does not contain your SMPL model.

In [ ]:
from pathlib import Path
import json, numpy as np

INPUT = Path('/kaggle/input')
frame = next((p for p in INPUT.rglob('000001.jpg') if p.parent.name == 'img1'), None)
TRACK_JSON = next(iter(INPUT.rglob('SNGS-043_football-player-detection-v9_botsort.json')), None)
SMPL_SOURCE = next(iter(INPUT.rglob('basicmodel_m_lbs_10_207_0_v1.1.0.pkl')), None)
assert frame and TRACK_JSON and SMPL_SOURCE, 'Attach SoccerNet frames + raw track JSON + private SMPL input'
FRAMES = frame.parent
TARGETS = [284, 17, 171]  # stable tracks that remain near the goal at frame 625
START, END = 375, 625    # clip time 00:00:15 through 00:00:25
WORK = Path('/kaggle/working/football3d_sngs043_approach')
WORK.mkdir(parents=True, exist_ok=True)
print('targets:', TARGETS, '| frames:', START, 'to', END, '| inputs:', len(list(FRAMES.glob('*.jpg'))))

In [ ]:
import shutil, subprocess, sys

REPO = Path('/kaggle/working/4D-Humans')
if not (REPO / 'hmr2' / '__init__.py').is_file():
    if REPO.exists(): shutil.rmtree(REPO)
    subprocess.run(['git', 'clone', '-q', 'https://github.com/shubham-goel/4D-Humans.git', str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for source in REPO.rglob('*.py'):
    text = source.read_text()
    if 'timm.models.layers' in text: source.write_text(text.replace('timm.models.layers', 'timm.layers'))
if not shutil.which('aria2c'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'aria2'], check=True)
sys.path.insert(0, str(REPO))

In [ ]:
import os, tarfile
from hmr2.configs import CACHE_DIR_4DHUMANS
from hmr2.models import DEFAULT_CHECKPOINT

CACHE, checkpoint = Path(CACHE_DIR_4DHUMANS), Path(DEFAULT_CHECKPOINT)
relative = checkpoint.relative_to(CACHE).as_posix()
SAVE_CACHE = Path('/kaggle/working/hmr2_cache')
def cache_root(folder):
    found = next((p for p in folder.rglob(checkpoint.name) if p.as_posix().endswith(relative)), None)
    return Path(found.as_posix()[:-len(relative)]) if found else None

# Reuse an attached private cache, or recover the earlier buggy extraction in WORK.
source = cache_root(INPUT) or cache_root(SAVE_CACHE) or cache_root(WORK)
if source is not None and not checkpoint.exists():
    shutil.copytree(source, CACHE, dirs_exist_ok=True, copy_function=os.symlink)
if not checkpoint.exists():
    SAVE_CACHE.mkdir(parents=True, exist_ok=True)
    archive = SAVE_CACHE / 'hmr2_data.tar.gz'
    subprocess.run(['aria2c', '--continue=true', '-x', '16', '-s', '16', '-k', '1M', '--file-allocation=none', '-d', str(SAVE_CACHE), '-o', archive.name, 'https://www.cs.utexas.edu/~pavlakos/4dhumans/hmr2_data.tar.gz'], check=True)
    assert not Path(str(archive) + '.aria2').exists(), 'Download incomplete: rerun this cell'
    with tarfile.open(archive, 'r:*') as bundle: bundle.extractall(SAVE_CACHE, filter='data')
    archive.unlink()
    source = cache_root(SAVE_CACHE)
    assert source is not None, f'checkpoint not found under {SAVE_CACHE}'
    shutil.copytree(source, CACHE, dirs_exist_ok=True, copy_function=os.symlink)
assert checkpoint.exists(), f'missing checkpoint: {checkpoint}'
for folder in [CACHE, *CACHE.rglob('*')]:
    if folder.is_dir() and not folder.is_symlink(): folder.chmod(0o755)
cache_model = CACHE / 'data/smpl/SMPL_NEUTRAL.pkl'
cache_model.parent.mkdir(parents=True, exist_ok=True)
cache_model.unlink(missing_ok=True)  # never save the licensed SMPL file in hmr2_cache
shutil.copy2(SMPL_SOURCE, cache_model)
print('checkpoint:', checkpoint)

In [ ]:
import cv2, torch
from hmr2.models import load_hmr2
from hmr2.datasets.vitdet_dataset import ViTDetDataset
from hmr2.utils import recursive_to

assert torch.cuda.is_available(), 'Enable a T4 GPU in Kaggle settings'
track = json.loads(TRACK_JSON.read_text())
boxes = {f['frame']: {b['id']: b['xyxy'] for b in f['boxes']} for f in track['frames']}
original_load = torch.load
def trusted_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return original_load(*args, **kwargs)
torch.load = trusted_load
try: model, config = load_hmr2(str(checkpoint))
finally: torch.load = original_load
model = model.cuda().eval()

for target in TARGETS:
    saved = {key: [] for key in ['frame', 'joints', 'global_orient_rotmat', 'body_pose_rotmat', 'betas', 'pred_cam', 'pred_cam_t']}
    for frame in range(START, END + 1):
        box = boxes.get(frame, {}).get(target)
        if box is None: continue
        image = cv2.imread(str(FRAMES / f'{frame:06d}.jpg'))
        dataset = ViTDetDataset(config, image, np.asarray(box, dtype=np.float32)[None])
        batch = recursive_to(next(iter(torch.utils.data.DataLoader(dataset, batch_size=1))), 'cuda')
        with torch.inference_mode(): result = model(batch)
        params = result['pred_smpl_params']
        male = model.smpl(global_orient=params['global_orient'].float(), body_pose=params['body_pose'].float(), betas=torch.zeros_like(params['betas']).float(), pose2rot=False)
        joints = torch.einsum('jk,kv->jv', model.smpl.J_regressor.cuda(), male.vertices[0])
        saved['frame'].append(frame); saved['joints'].append(joints.cpu().numpy())
        saved['global_orient_rotmat'].append(params['global_orient'][0].cpu().numpy())
        saved['body_pose_rotmat'].append(params['body_pose'][0].cpu().numpy())
        saved['betas'].append(params['betas'][0].cpu().numpy())
        saved['pred_cam'].append(result['pred_cam'][0].cpu().numpy())
        saved['pred_cam_t'].append(result['pred_cam_t'][0].cpu().numpy())
    out = WORK / f'pose_{target}.npz'
    np.savez_compressed(out, **{key: np.asarray(value) for key, value in saved.items()}, fps=np.float32(25))
    print('saved', out.name, 'frames:', len(saved['frame']))

In [ ]:
from IPython.display import FileLink, display
for target in TARGETS: display(FileLink(str(WORK / f'pose_{target}.npz')))